# 00 — Setup

Run once after `infra/deploy.sh` to check that every resource is reachable and the corpus is in Blob Storage.


In [ ]:
import sys
sys.path.insert(0, "..")

from config import load_config

config = load_config()
print("Config loaded OK. Resolved endpoints:")
print(" search:       ", config.search_endpoint)
print(" storage:      ", config.storage_account_url)
print(" openai:       ", config.openai_endpoint)
print(" ai services:  ", config.ai_services_endpoint or "(not set — only needed for OCR in 02)")


## 1. Azure AI Search reachable, Entra ID auth working

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.search.documents.indexes import SearchIndexClient

credential = DefaultAzureCredential()
index_client = SearchIndexClient(config.search_endpoint, credential)

existing = [i.name for i in index_client.list_indexes()]
print(f"Connected. {len(existing)} index(es) currently on the service:", existing or "(none yet)")


## 2. Blob container exists, then upload the corpus

One folder per state, matching `data/README.md`.


In [ ]:
from pathlib import Path

from azure.storage.blob import BlobServiceClient

STATE_FILES = {
    "california": "california-driver-handbook.pdf",
    "florida": "florida-driver-handbook.pdf",
    "new-jersey": "newjersey-driver-manual.pdf",
    "virginia": "virginia-drivers-manual.pdf",
    "wisconsin": "wisconsin-motorists-handbook.pdf",
}

blob_service = BlobServiceClient(config.storage_account_url, credential=credential)
container_client = blob_service.get_container_client(config.storage_container_name)
if not container_client.exists():
    container_client.create_container()
    print(f"Created container '{config.storage_container_name}'.")
else:
    print(f"Container '{config.storage_container_name}' already exists.")


### Split PDFs over the indexer's 16 MB limit

On the Basic tier, the indexer silently skips any document over 16 MB, so larger PDFs are split before upload.


In [ ]:
import io

from pypdf import PdfReader, PdfWriter

MAX_INDEXER_DOC_BYTES = 14 * 1024 * 1024  # 14 MB — safety margin under the 16 MB limit


def split_oversized_pdf(path):
    """Yield (part_name, bytes): the file as is if it fits, else page-range halves under the limit."""
    reader = PdfReader(path)
    n = len(reader.pages)

    def render(first, last):
        writer = PdfWriter()
        for p in reader.pages[first:last + 1]:
            writer.add_page(p)
        buf = io.BytesIO()
        writer.write(buf)
        return buf.getvalue()

    if path.stat().st_size <= MAX_INDEXER_DOC_BYTES:
        yield path.name, path.read_bytes()
        return

    ranges = []

    def split_range(first, last):
        data = render(first, last)
        if len(data) <= MAX_INDEXER_DOC_BYTES or first == last:
            ranges.append((first, last, data))
            return
        mid = (first + last) // 2
        split_range(first, mid)
        split_range(mid + 1, last)

    split_range(0, n - 1)
    ranges.sort()
    stem = path.stem
    for i, (_, _, data) in enumerate(ranges, start=1):
        yield f"{stem}-part{i:02d}.pdf", data


In [ ]:
data_dir = Path("../data")

for state, filename in STATE_FILES.items():
    local_path = data_dir / filename
    if not local_path.exists():
        print(f"MISSING locally: {local_path} — run data/download.sh first.")
        continue
    for part_name, part_bytes in split_oversized_pdf(local_path):
        blob_name = f"{state}/{part_name}"
        blob_client = container_client.get_blob_client(blob_name)
        if blob_client.exists():
            print(f"Already uploaded: {blob_name}")
            continue
        blob_client.upload_blob(part_bytes, overwrite=False)
        print(f"Uploaded: {blob_name} ({len(part_bytes)/1024/1024:.1f} MB)")


In [ ]:
uploaded = sorted(b.name for b in container_client.list_blobs())
print(f"{len(uploaded)} blob(s) in '{config.storage_container_name}':")
for name in uploaded:
    print(" -", name)

states_present = {name.split("/")[0] for name in uploaded}
missing = set(STATE_FILES) - states_present
assert not missing, f"No blobs uploaded for: {missing} — check the upload step above."


## 3. Embedding deployment reachable

Checks that the embedding deployment in `.env` resolves.


In [ ]:
from azure.identity import get_bearer_token_provider
from openai import AzureOpenAI

token_provider = get_bearer_token_provider(
    credential, "https://cognitiveservices.azure.com/.default"
)
openai_client = AzureOpenAI(
    azure_endpoint=config.openai_endpoint,
    azure_ad_token_provider=token_provider,
    api_version="2024-10-21",
)

response = openai_client.embeddings.create(
    input="What's the speed limit in a school zone?",
    model=config.openai_embedding_deployment,
)
print(f"Embedding OK — {len(response.data[0].embedding)} dimensions.")


## 4. Ready

If every cell above ran clean, you can start `module-2/02_indexing_pipeline.ipynb`.
